# Limpieza y Preprocesamiento del Dataset Bank Marketing

Este notebook documenta la limpieza, validación y transformación del dataset **Bank Marketing** (campañas entre mayo de 2008 y noviembre de 2010). El propósito es dejar los datos listos para **análisis exploratorio posterior**, no para entrenar ni evaluar modelos predictivos (no se realiza train/test split ni modelado).


## 1. Configuración e imports

Se agrega la raíz del proyecto a `sys.path` para importar los módulos de `src/`.


In [ ]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.load_data import (
    get_outputs_dir,
    get_processed_dir,
    get_project_root,
    load_bank_data,
)
from src.validation import (
    calcular_checksum_md5,
    count_unknown_values,
    get_basic_summary,
    validate_columns,
)
from src.cleaning import (
    cap_outliers_iqr,
    detect_outliers_iqr,
    drop_leakage_columns,
    impute_categorical_mode,
    impute_pdays_clean_median,
    replace_unknown_with_nan,
    transform_pdays,
)
from src.transform import encode_target, one_hot_encode, scale_numeric_columns
from src.visualization import plot_numeric_boxplots, plot_target_distribution

pd.options.display.max_columns = 50


## 2. Revisión de la estructura del proyecto

Antes de cargar datos se verifica que existan carpetas clave, el CSV raw y los módulos en `src/`.


In [ ]:

PROJECT_ROOT = get_project_root()
print("Raíz del proyecto:", PROJECT_ROOT.resolve())

expected_dirs = ["data", "data/raw", "notebooks", "outputs", "src"]
print("\nCarpetas esperadas:")
for d in expected_dirs:
    p = PROJECT_ROOT / Path(d)
    ok = p.is_dir()
    label = "OK" if ok else "FALTA"
    print(f"  {label}  {p.relative_to(PROJECT_ROOT)}")

raw_csv = PROJECT_ROOT / "data" / "raw" / "bank-additional-full.csv"
print("\nDataset raw:", "existe" if raw_csv.exists() else "NO ENCONTRADO", raw_csv.relative_to(PROJECT_ROOT))

src_files = [
    "load_data.py",
    "validation.py",
    "cleaning.py",
    "transform.py",
    "visualization.py",
]
print("\nMódulos src/:")
for name in src_files:
    p = PROJECT_ROOT / "src" / name
    label = "OK" if p.is_file() else "FALTA"
    print(f"  {label}  src/{name}")

PROCESSED_DIR = get_processed_dir()
OUTPUTS_DIR = get_outputs_dir()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = OUTPUTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## 3. Carga de datos

Lectura con separador `;` y validación de existencia del archivo (implementado en `load_bank_data`).


In [ ]:
df = load_bank_data()
df.head()


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.describe(include="object")


## 4. Validación de columnas

Se contrastan las columnas del archivo con el esquema esperado del caso Bank Marketing.


In [ ]:
col_report = validate_columns(df)
col_report


## 5. Diagnóstico inicial de calidad

Nulos nativos, valores `unknown`, duplicados y distribución de la respuesta `y`.


In [ ]:

summary = get_basic_summary(df)
pd.Series(summary["null_values"], name="nulos")


In [ ]:
unknown_df = count_unknown_values(df)
unknown_df


In [ ]:
print("Filas duplicadas:", int(df.duplicated().sum()))


In [ ]:
plot_target_distribution(df, save_path=FIGURES_DIR / "distribucion_y.png")


## 6. Tratamiento de valores `unknown`

Según el diccionario del dataset, `unknown` representa información no declarada. Se reemplaza por `NaN` **solo en columnas donde aparece** (`replace_unknown_with_nan`), sin alterar otros textos. Luego se imputan las categóricas con la **moda** (`impute_categorical_mode`), estrategia simple que preserva el número de filas (no se eliminan registros masivamente).

Columnas donde suele aparecer `unknown`: job, marital, education, default, housing, loan.


In [ ]:

unknown_before = count_unknown_values(df)
df_step = replace_unknown_with_nan(df)
unknown_after_unknown_removed = count_unknown_values(df_step)
cat_na_before = df_step.select_dtypes(include="object").isna().sum()
df_step = impute_categorical_mode(df_step)
cat_na_after = df_step.select_dtypes(include="object").isna().sum()

display("unknown antes (conteo):", unknown_before)
display("Tras replace_unknown: columnas con 'unknown' restante:", unknown_after_unknown_removed)
display("Nulos en categóricas antes de imputar moda:", cat_na_before[cat_na_before > 0])
display("Nulos en categóricas después de imputar moda:", cat_na_after[cat_na_after > 0])


## 7. Tratamiento especial de `pdays`

En este dataset, **`pdays == 999` significa que el cliente no fue contactado previamente**, no es un outlier numérico arbitrario.

Se crean:

- `was_previously_contacted`: 0 si `pdays == 999`, 1 en caso contrario.
- `pdays_clean`: mismos valores que `pdays` pero con 999 reemplazado por `NaN`.

Después se imputa `pdays_clean` con la **mediana** de los días reales de contacto previo para evitar nulos en etapas posteriores. Finalmente se eliminará la columna original `pdays` junto con `duration`.


In [ ]:

df_step = transform_pdays(df_step)
df_step[["pdays", "was_previously_contacted", "pdays_clean"]].head(12)


In [ ]:

df_step = impute_pdays_clean_median(df_step)
df_step["pdays_clean"].isna().sum()


## 8. Análisis de `duration` (fuga de información)

`duration` es la duración del último contacto en segundos; se conoce **después** de la llamada y puede relacionarse directamente con la suscripción al depósito. Por ello introduce **data leakage** en un escenario predictivo realista.

En esta entrega solo se describe y **no se usará como variable final**: se elimina del dataset limpio junto con `pdays` original.


In [ ]:
df_step["duration"].describe()


## 9. Análisis de outliers (IQR)

Se calculan límites por rango intercuartílico para las variables numéricas indicadas en la guía.

**Criterios:**

- No se trata `pdays == 999` como error de medición; ya fue separado en `pdays_clean` y `was_previously_contacted`.
- No se aplica **capping** a `pdays` original (código especial 999).
- No se capea `duration` (se eliminará por fuga).
- Los indicadores macroeconómicos reflejan condiciones reales del periodo: se **reportan** outliers pero **no se recortan**; más adelante solo se **escalan**.

**Capping IQR aplicado solo a:** `age`, `campaign`, `previous`, `pdays_clean`.


In [ ]:

numeric_eda_cols = [
    "age",
    "campaign",
    "pdays",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
    "duration",
]
outliers_tbl = detect_outliers_iqr(df_step, numeric_eda_cols)
outliers_tbl


In [ ]:

cols_to_plot_before = [
    c for c in ["age", "campaign", "previous", "pdays_clean", "duration"] if c in df_step.columns
]
plot_numeric_boxplots(df_step, cols_to_plot_before, save_prefix=FIGURES_DIR / "box_before")


In [ ]:

cap_cols = [c for c in ["age", "campaign", "previous", "pdays_clean"] if c in df_step.columns]
df_step = cap_outliers_iqr(df_step, cap_cols)
plot_numeric_boxplots(df_step, cap_cols, save_prefix=FIGURES_DIR / "box_after_cap")


## 10. Eliminación de columnas problemáticas

Se eliminan `duration` (fuga) y `pdays` (código especial sustituido por `pdays_clean` + `was_previously_contacted`).


In [ ]:

df_step = drop_leakage_columns(df_step)
assert "duration" not in df_step.columns
assert "pdays" not in df_step.columns
df_step.head()


## 11. Codificación de la variable objetivo

`yes` → 1, `no` → 0. Se valida que no queden nulos por mapeo.


In [ ]:

df_clean = encode_target(df_step)
assert df_clean["y"].notna().all()
df_clean["y"].value_counts()


## 12. Codificación one-hot de categóricas

Las variables texto restantes se convierten en columnas binarias con `pd.get_dummies(..., drop_first=True)`, lo que evita redundancia por la **trampa de la variable dummy** y prepara datos numéricamente homogéneos para análisis.

Categóricas esperadas en el caso: job, marital, education, default, housing, loan, contact, month, day_of_week, poutcome.


In [ ]:

obj_cols = df_clean.select_dtypes(include="object").columns.tolist()
print("Categóricas antes de one-hot:", obj_cols)
df_clean = one_hot_encode(df_clean, target_col="y")
df_clean.head()


## 13. Escalamiento de variables numéricas

Se aplica `StandardScaler` a todas las columnas numéricas **excepto** `y`, de modo que edad, contactos e indicadores económicos queden en escala comparable. **No se escala la variable objetivo.** La función devuelve también el objeto `StandardScaler` ajustado (`scaler_bank`) para documentar y posiblemente reutilizar la misma transformación sobre datos nuevos.


In [ ]:
df_clean, scaler_bank = scale_numeric_columns(df_clean, target_col="y")
# scaler_bank conserva media y escala aprendidas (reutilizable sobre nuevas filas con las mismas columnas).


## 14. Dataset final y comprobaciones

Se verifica ausencia de nulos, ausencia de `unknown`, y que `duration` y `pdays` no estén presentes.


In [ ]:

display(df_clean.head())
print("Shape final:", df_clean.shape)
df_clean.info()


In [ ]:

null_total = df_clean.isna().sum().sum()
unknown_mask = df_clean.astype(str).eq("unknown")
unknown_total = unknown_mask.sum().sum()
print("Total nulos:", int(null_total))
print("Total celdas con texto 'unknown':", int(unknown_total))
print("duration en columnas:", "duration" in df_clean.columns)
print("pdays en columnas:", "pdays" in df_clean.columns)


## 15. Guardado de resultados

- Dataset limpio: `data/processed/bank_cleaned.csv`
- Resumen tabular: `outputs/limpieza_resumen.csv`
- Reporte narrativo: `outputs/limpieza_resumen.md`

No se modifica el archivo original en `data/raw/`.


In [ ]:

out_csv = PROCESSED_DIR / "bank_cleaned.csv"
df_clean.to_csv(out_csv, index=False)
print("Guardado:", out_csv.relative_to(PROJECT_ROOT))

resumen_rows = [
    {"metrica": "dataset", "valor": "bank-additional-full.csv"},
    {"metrica": "ruta_original", "valor": str(raw_csv.relative_to(PROJECT_ROOT))},
    {"metrica": "filas_inicial", "valor": df.shape[0]},
    {"metrica": "columnas_inicial", "valor": df.shape[1]},
    {"metrica": "filas_final", "valor": df_clean.shape[0]},
    {"metrica": "columnas_final", "valor": df_clean.shape[1]},
    {"metrica": "duplicados_inicial", "valor": summary["duplicated_rows"]},
    {"metrica": "columnas_con_unknown_inicial", "valor": len(unknown_df)},
    {"metrica": "estrategia_faltantes", "valor": "unknown->NaN; moda en categóricas; mediana en pdays_clean"},
    {"metrica": "estrategia_pdays", "valor": "999->NaN en pdays_clean; was_previously_contacted; eliminar pdays"},
    {"metrica": "duration", "valor": "Eliminada por fuga de información"},
    {"metrica": "outliers", "valor": "IQR cap: age,campaign,previous,pdays_clean; macros solo escalados"},
    {"metrica": "codificacion", "valor": "y 0/1; get_dummies drop_first en categóricas"},
    {"metrica": "escalado", "valor": "StandardScaler en numéricas excepto y"},
    {"metrica": "archivo_procesado", "valor": str(out_csv.relative_to(PROJECT_ROOT))},
]
resumen_df = pd.DataFrame(resumen_rows)
resumen_path = OUTPUTS_DIR / "limpieza_resumen.csv"
resumen_df.to_csv(resumen_path, index=False)
print("Resumen CSV:", resumen_path.relative_to(PROJECT_ROOT))


In [ ]:
md_path = OUTPUTS_DIR / "limpieza_resumen.md"
checksum = calcular_checksum_md5(raw_csv)

unknown_md = (
    "```\n" + unknown_df.to_string() + "\n```" if len(unknown_df) else "_Ninguna detectada._"
)
n_num_predictoras = len([c for c in df_clean.columns if c != "y"])
r_raw = raw_csv.relative_to(PROJECT_ROOT).as_posix()
r_out = out_csv.relative_to(PROJECT_ROOT).as_posix()
r_sum = resumen_path.relative_to(PROJECT_ROOT).as_posix()

reporte = "\n".join(
    [
        "# Reporte de limpieza — Bank Marketing",
        "",
        "## Dataset",
        "- **Nombre:** bank-additional-full.csv",
        "- **Ruta original (relativa):** `{}`".format(r_raw),
        "- **Checksum MD5 (raw):** `{}`".format(checksum),
        "",
        "## Dimensiones",
        "| Etapa | Filas | Columnas |",
        "|-------|-------|----------|",
        "| Inicial | {} | {} |".format(df.shape[0], df.shape[1]),
        "| Final | {} | {} |".format(df_clean.shape[0], df_clean.shape[1]),
        "",
        "## Columnas con `unknown` (inicial)",
        unknown_md,
        "",
        "## Estrategias",
        "1. **Valores faltantes (`unknown`):** sustitución por NaN solo donde aparece la cadena `unknown`; imputación de categóricas con **moda**.",
        '2. **`pdays`:** valor **999** tratado como "no contactado"; variables derivadas `was_previously_contacted` y `pdays_clean`; imputación de `pdays_clean` con **mediana**; eliminación de `pdays` original.',
        "3. **`duration`:** eliminada por **fuga de información** respecto del resultado de la llamada.",
        "4. **Outliers:** detección IQR en variables numéricas; **capping** aplicado solo a `age`, `campaign`, `previous`, `pdays_clean`. Indicadores macro **no capeados** (contexto económico real); solo escalados al final.",
        "5. **Codificación:** `y` como 0/1; demás categóricas con **one-hot** (`drop_first=True`).",
        "6. **Escalamiento:** **StandardScaler** en todas las columnas numéricas excepto `y`.",
        "",
        "## Variables escaladas (todas las numéricas excepto `y`)",
        "Total columnas predictoras numéricas (excluye objetivo y): {}.".format(n_num_predictoras),
        "",
        "## Archivos generados",
        "- `{}`".format(r_out),
        "- `{}`".format(r_sum),
        "- Figuras opcionales en `outputs/figures/`",
    ]
)

md_path.write_text(reporte, encoding="utf-8")
print("Reporte Markdown:", md_path.relative_to(PROJECT_ROOT))
